# 🚀 AI Multi-Docs Extraction Pipeline: Walkthrough & Real Data Runner

สมุดบันทึก (Jupyter Notebook) สำหรับรันกระบวนการสกัดและประมวลผลเอกสารทั้งระบบแบบ **End-to-End ด้วยข้อมูลจริงจาก `pipeline_storage`**
ช่วยให้สามารถรันและตรวจสอบการไหลของข้อมูลจริงในแต่ละขั้นตอน (Step-by-Step Execution & Observability) ได้อย่างสมบูรณ์

## 🛠️ Step 0: ตั้งค่า Working Directory, Environment และนำเข้า Pipeline Services

In [ ]:
import os
import sys
import glob
import json
import sqlite3
import pandas as pd
from IPython.display import display, Image, JSON
from dotenv import load_dotenv

# 1. ปรับ Working Directory และ sys.path ให้ชี้ไปที่ Root ของโปรเจกต์เสมอ
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📂 Project Root Working Directory: {os.getcwd()}")

# 2. โหลด Environment Variables และ Pipeline Services
load_dotenv()
from src.core.pipeline import (
    run_init,
    run_split_and_match,
    run_extract,
    run_validate,
    run_transform_to_db,
    run_export_outputs,
    run_pipeline_all,
    reset_pipeline_data
)
from src.core.db import get_db_connection
from src.core.config_loader import load_system_settings, get_default_domain

DOMAIN = get_default_domain()  # ค่าเริ่มต้น: 'expense_receipt'
print(f"✅ Pipeline Services Ready. Active Target Domain: '{DOMAIN}'")

## 🧹 Step 0.1: (Optional) รีเซ็ตฐานข้อมูลและล้างไฟล์ชั่วคราว (Fresh Start)
กดรันเซลล์นี้เมื่อต้องการ **ล้างประวัติเอกสารใน SQLite** และ **ล้างไฟล์ชั่วคราวใน `02_split_pages/` และ `03_processing_queue/`** เพื่อเริ่มทดสอบใหม่ตั้งแต่ต้น

In [ ]:
# ปลดล็อคหรือกดรันเซลล์นี้เพื่อเคลียร์ข้อมูลเดิมก่อนเริ่มรันใหม่
reset_result = reset_pipeline_data(domain=DOMAIN, clear_storage_temp=True, clear_database=True)
print("🧹 Reset Pipeline Data Result:", reset_result)
print("🎉 Ready for a brand new clean run!")

## ⚙️ Step 1: System Initialization (`Run_01`)
ตรวจสอบความพร้อมของ `settings.json`, Schema ฐานข้อมูล SQLite, และโครงสร้างโฟลเดอร์ใน `pipeline_storage`

In [ ]:
print("--- [Stage 1] Initializing System & Validating Environment ---")
init_success = run_init()
if init_success:
    print("🎉 System is READY and all storage folders & DB tables are verified!")
else:
    print("❌ System initialization encountered errors. Please check configs or .env")

## 📄 Step 2: Split PDFs & Match Merchant Sources (`Run_02`)
อ่านไฟล์เอกสารจริงจาก `pipeline_storage/expense_receipt/01_raw_inbox/` (รองรับทั้ง PDF และภาพ `.jpg`, `.png`, `.webp`) เพื่อ:
1. ตรวจจับร้านค้า (Merchant Matching: SPX, Grab, Shopee)
2. ตัดหน้า PDF เป็นไฟล์ภาพ `.jpg` (คุณภาพ 85% คุม Max Dimension 1800px) ลงใน `02_split_pages/`
3. ลงทะเบียน Batch & Pages เข้าสู่ฐานข้อมูล

In [ ]:
print("--- [Stage 2] Processing Inbox Documents and Matching Merchant Sources ---")
split_results = run_split_and_match(domain=DOMAIN)

if split_results:
    print(f"\n✅ Successfully processed {len(split_results)} document batch(es):")
    for res in split_results:
        print(f"\n📦 Batch ID: {res['batch_id']}")
        print(f"   - Original File: {res['filename']}")
        print(f"   - Matched Merchant: {res['matched_source']}")
        print(f"   - Total Pages: {res['total_pages']}")
        print(f"   - Split Page Images:")
        for img in res['page_images']:
            print(f"     🖼️ {img}")
            if os.path.exists(img):
                display(Image(filename=img, width=400))
else:
    print("ℹ️ No new document files found in raw inbox. (You can place PDF/JPG files into pipeline_storage/expense_receipt/01_raw_inbox/)")

## 🤖 Step 3: AI Document Extraction (`Run_03`)
นำรูปภาพหน้าที่ตัดแล้วใน `02_split_pages/` ส่งให้ AI (Gemini / OpenAI) สกัดข้อมูลตามโครงสร้าง `schema.json`
และบันทึกไฟล์ JSON ที่สกัดได้ลงใน `03_processing_queue/{merchant}/`

In [ ]:
print("--- [Stage 3] Extracting Document Data with AI ---")
extract_result = run_extract(domain=DOMAIN)
print(f"📊 Extraction Summary: {extract_result}")

# แสดงตัวอย่างไฟล์ JSON ที่สกัดได้ในคิว
queue_pattern = f"pipeline_storage/{DOMAIN}/03_processing_queue/**/*.json"
queue_files = glob.glob(queue_pattern, recursive=True)
if queue_files:
    print(f"\n💾 Found {len(queue_files)} extracted JSON file(s) in queue:")
    sample_file = queue_files[0]
    print(f"   Showing preview from: {sample_file}")
    with open(sample_file, "r", encoding="utf-8") as jf:
        sample_json = json.load(jf)
    display(JSON(sample_json))
else:
    print("ℹ️ No queue JSON files found.")

## 🛡️ Step 4: Validate & Post-Process Data (`Run_04`)
ตรวจสอบความถูกต้องของข้อมูลจริงใน `03_processing_queue/`:
- ตรวจสอบเลขประจำตัวผู้เสียภาษี (Tax ID) ตามกฎร้านค้า
- ปรับรูปแบบวันที่ปี พ.ศ. ➔ ค.ศ. (BE to AD Normalization)
- ตรวจสอบสูตรการเงิน (Subtotal - Discount + VAT == Net)
- ตรวจสอบผลรวมรายการสินค้าเทียบกับ Subtotal
- จัดลำดับความสำคัญ (Review Priority: HIGH/MED/LOW) และกำหนดสถานะ `PROCESSED` หรือ `NEEDS_REVIEW`

In [ ]:
print("--- [Stage 4] Validating and Post-Processing Queue Records ---")
validate_result = run_validate(domain=DOMAIN)
print(f"📊 Validation Summary: {validate_result}")

## 💾 Step 5: Transform Data to Relational SQLite Database (`Run_05`)
นำเข้าข้อมูลที่ผ่านการตรวจสอบแล้วเข้าสู่ฐานข้อมูล SQLite ในตาราง `documents`, `expense_receipts`, และ `receipt_items`

In [ ]:
print("--- [Stage 5] Importing Records into Relational Database ---")
db_result = run_transform_to_db(domain=DOMAIN)
print(f"📊 DB Transformation Summary: {db_result}")

# Query ตรวจสอบข้อมูลในตาราง SQLite แบบ Real-time ด้วย Pandas
conn = get_db_connection()
df_docs = pd.read_sql_query("""
    SELECT document_id, domain_id, source_id, status_code, doc_number, doc_date, entity_name, total_amount, created_at 
    FROM documents 
    ORDER BY created_at DESC 
    LIMIT 10
""", conn)

df_receipts = pd.read_sql_query("""
    SELECT receipt_id, document_id, merchant_name, tax_id, subtotal, vat_amount, net_amount, payment_method 
    FROM expense_receipts 
    ORDER BY receipt_id DESC 
    LIMIT 10
""", conn)

df_items = pd.read_sql_query("""
    SELECT item_id, receipt_id, item_name, quantity, unit, unit_price, total_price 
    FROM receipt_items 
    ORDER BY item_id DESC 
    LIMIT 10
""", conn)
conn.close()

print("\n📊 Documents Master Table (Last 10 records):")
display(df_docs)
print("\n📊 Expense Receipts Table (Last 10 records):")
display(df_receipts)
print("\n📊 Receipt Items Table (Last 10 records):")
display(df_items)

## 📊 Step 6: Generate Output Reports (`Run_06`)
สร้างรายงานสรุปผลลัพธ์ผ่าน Exporters ทั้งหมด (Google Sheet Summary, Accounting Line Items, Express PV Voucher)

In [ ]:
print("--- [Stage 6] Generating Output Reports (CSV / Excel / Express PV) ---")
export_result = run_export_outputs(domain=DOMAIN)
print(f"📊 Export Summary: {export_result}")

# แสดงผลลัพธ์รายงานที่ Export ออกมาในโฟลเดอร์ outputs/
csv_exports = glob.glob("outputs/*.csv")
for csv_file in csv_exports:
    print(f"\n--------------------------------------------------------")
    print(f"📄 Output Report: {csv_file}")
    print(f"--------------------------------------------------------")
    try:
        encoding = "cp874" if "express_pv" in csv_file else "utf-8-sig"
        df_report = pd.read_csv(csv_file, encoding=encoding)
        display(df_report.head(10))
    except Exception as read_err:
        print(f"Could not preview CSV: {read_err}")

## ⚡ Step 7: (Optional) Full Pipeline Single-Command Execution (`Run All`)
รันทุกขั้นตอนตั้งแต่ต้นจนจบ (Stage 1 ➔ Stage 6) ในคำสั่งเดียว

In [ ]:
# ปลด comment เมื่อต้องการสั่งรันทุก Stage พร้อมกันแบบ One-Shot
# print("🚀 Executing Full Pipeline End-to-End...")
# full_pipeline_result = run_pipeline_all(domain=DOMAIN)
# print("🎉 Pipeline Run Completed:", full_pipeline_result)